In [1]:
# =================================================================================================
# NHANES 2005-2006 LAB MERGE - FIDELITY PRESERVING
#
# Purpose:
#   Merge corrected NHANES laboratory CSV files into one participant-level Lab CSV.
#
# Source files:
#   GLU_D_corrected.csv
#   TCHOL_D_corrected.csv
#   FASTQX_D_corrected.csv
#   GHB_D_corrected.csv
#   HDL_D_corrected.csv
#   TRIGLY_D_corrected.csv
#
# Fidelity rules:
#
#   1. SEQN remains character/integer-looking without ".0"
#
#   2. All corrected Lab decimal text is preserved AS IS.
#      No rounding.
#      No float_format=.
#      No conversion of Lab measurements to integer.
#
#   3. WTSAF2YR occurs in GLU_D and TRIGLY_D.
#      Validate numeric equality first.
#      Keep canonical WTSAF2YR from GLU_D.
#      Remove redundant TRIGLY_D copy BEFORE merge.
#
#   4. PHAFSTHR / PHAFSTMN occur in GLU_D and FASTQX_D.
#
#      FASTQX_D has broader participant coverage, therefore both
#      source versions are retained:
#
#          PHAFSTHR
#          PHAFSTMN
#
#          PHAFSTHR_FAST
#          PHAFSTMN_FAST
#
#      On overlapping observations, values are compared NUMERICALLY.
#
#      Example:
#
#          "8" and "8.0"
#
#      are numerically equivalent and should NOT cause a failure.
#
#   5. Every source file must contain unique SEQN.
#
#   6. Every merge uses validate="one_to_one".
#
#   7. Exact source text is checked after merge for all retained
#      source-specific variables.
#
# =================================================================================================


# =================================================================================================
# 1. IMPORTS
# =================================================================================================

import pandas as pd
import numpy as np
from pathlib import Path


# =================================================================================================
# 2. PATHS
# =================================================================================================

BASE_DIR = Path().resolve().parent


LAB_DIR = (
    BASE_DIR
    / "data"
    / "2013_2014"
    / "Labs"
)


GLU_FILE = (
    LAB_DIR
    / "GLU_H_corrected.csv"
)


TCHOL_FILE = (
    LAB_DIR
    / "TCHOL_H_corrected.csv"
)


FASTQX_FILE = (
    LAB_DIR
    / "FASTQX_H_corrected.csv"
)


GHB_FILE = (
    LAB_DIR
    / "GHB_H_corrected.csv"
)


HDL_FILE = (
    LAB_DIR
    / "HDL_H_corrected.csv"
)


TRIGLY_FILE = (
    LAB_DIR
    / "TRIGLY_H_corrected.csv"
)


OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "processed_v3"
    / "2013_2014"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


OUTPUT_PATH = (
    OUTPUT_DIR
    / "neighbourhood_nhanes_labs_2013_2014_raw_corrected.csv"
)


# =================================================================================================
# 3. READ CORRECTED CSV FILES
# =================================================================================================

#
# IMPORTANT:
#
# dtype=str
#
# prevents pandas from reformatting already-corrected Lab decimals
# during this merge operation.
#

glucose = pd.read_csv(
    GLU_FILE,
    dtype=str,
    keep_default_na=False
)


chol = pd.read_csv(
    TCHOL_FILE,
    dtype=str,
    keep_default_na=False
)


fasting = pd.read_csv(
    FASTQX_FILE,
    dtype=str,
    keep_default_na=False
)


ghb = pd.read_csv(
    GHB_FILE,
    dtype=str,
    keep_default_na=False
)


hdl = pd.read_csv(
    HDL_FILE,
    dtype=str,
    keep_default_na=False
)


trigly = pd.read_csv(
    TRIGLY_FILE,
    dtype=str,
    keep_default_na=False
)


# =================================================================================================
# 4. STANDARDIZE COLUMN NAMES
# =================================================================================================

datasets = {

    "GLU_D": glucose,

    "TCHOL_D": chol,

    "FASTQX_D": fasting,

    "GHB_D": ghb,

    "HDL_D": hdl,

    "TRIGLY_D": trigly

}


for name, dataset in datasets.items():

    dataset.columns = (
        dataset.columns
        .str.strip()
        .str.upper()
    )


# =================================================================================================
# 5. DISPLAY SOURCE FILE STRUCTURE
# =================================================================================================

print("=" * 110)

print(
    "NHANES 2011-2012 LAB MERGE"
)

print("=" * 110)


summary = []


for name, dataset in datasets.items():

    summary.append({

        "file": name,

        "rows": len(dataset),

        "columns": len(dataset.columns),

        "column_names": dataset.columns.tolist()

    })


summary_df = pd.DataFrame(
    summary
)


display(
    summary_df[
        [
            "file",
            "rows",
            "columns",
            "column_names"
        ]
    ]
)


for name, dataset in datasets.items():

    print(
        "\n"
        + "=" * 80
    )

    print(
        "FILE:",
        name
    )

    print(
        "ROWS:",
        f"{len(dataset):,}"
    )

    print(
        "COLUMNS:",
        len(dataset.columns)
    )

    print(
        dataset.columns.tolist()
    )


# =================================================================================================
# 6. SEQN VALIDATION FUNCTION
# =================================================================================================

def validate_seqn(
    dataset,
    dataset_name
):

    print(
        f"\n--- SEQN CHECK: {dataset_name} ---"
    )


    if "SEQN" not in dataset.columns:

        raise ValueError(
            f"{dataset_name}: SEQN is missing."
        )


    blank_count = int(
        dataset["SEQN"]
        .eq("")
        .sum()
    )


    duplicate_count = int(
        dataset["SEQN"]
        .duplicated()
        .sum()
    )


    dot_zero_count = int(
        (
            dataset["SEQN"].ne("")
            &
            dataset["SEQN"]
            .str.endswith(".0")
        )
        .sum()
    )


    invalid_format_count = int(
        (
            dataset["SEQN"].ne("")
            &
            ~dataset["SEQN"]
            .str.fullmatch(
                r"\d+"
            )
        )
        .sum()
    )


    print(
        "Rows:",
        f"{len(dataset):,}"
    )

    print(
        "Unique SEQN:",
        f"{dataset['SEQN'].nunique():,}"
    )

    print(
        "Blank SEQN:",
        blank_count
    )

    print(
        "Duplicate SEQN:",
        duplicate_count
    )

    print(
        "SEQN ending '.0':",
        dot_zero_count
    )

    print(
        "Invalid SEQN format:",
        invalid_format_count
    )


    if blank_count != 0:

        raise ValueError(
            f"{dataset_name}: blank SEQN found."
        )


    if duplicate_count != 0:

        raise ValueError(
            f"{dataset_name}: duplicate SEQN found."
        )


    if dot_zero_count != 0:

        raise ValueError(
            f"{dataset_name}: SEQN still contains '.0'."
        )


    if invalid_format_count != 0:

        raise ValueError(
            f"{dataset_name}: SEQN contains invalid characters."
        )


    print(
        "PASS"
    )


# =================================================================================================
# 7. VALIDATE SEQN IN EVERY SOURCE
# =================================================================================================

for name, dataset in datasets.items():

    validate_seqn(
        dataset,
        name
    )


print(
    "\nPASS: SEQN integrity validated in all six Lab files."
)


# =================================================================================================
# 8. SHOW ALL CROSS-FILE VARIABLE OVERLAPS
# =================================================================================================

print(
    "\n"
    + "=" * 110
)

print(
    "CROSS-FILE VARIABLE OVERLAPS"
)

print(
    "=" * 110
)


dataset_names = list(
    datasets.keys()
)


for i in range(
    len(dataset_names)
):

    for j in range(
        i + 1,
        len(dataset_names)
    ):

        left_name = (
            dataset_names[i]
        )

        right_name = (
            dataset_names[j]
        )


        left_columns = set(
            datasets[
                left_name
            ].columns
        )


        right_columns = set(
            datasets[
                right_name
            ].columns
        )


        overlap = sorted(

            (
                left_columns
                &
                right_columns
            )

            -

            {
                "SEQN"
            }

        )


        if overlap:

            print(

                f"{left_name} <-> {right_name}:",

                overlap

            )


# =================================================================================================
# 9. NUMERIC OVERLAP COMPARISON FUNCTION
# =================================================================================================

#
# IMPORTANT:
#
# The source CSVs are deliberately read as strings.
#
# Therefore raw comparison:
#
#     "8" != "8.0"
#
# would produce a false failure.
#
# This function:
#
#   - leaves original data unchanged
#   - compares numerical copies
#   - separately reports text-format differences
#

def compare_numeric_variable_by_seqn(
    left_df,
    right_df,
    column,
    left_name,
    right_name,
    require_same_seqn_set=False
):

    print(
        "\n"
        + "-" * 100
    )

    print(
        f"NUMERIC FIDELITY CHECK: "
        f"{left_name}.{column} "
        f"<-> "
        f"{right_name}.{column}"
    )

    print(
        "-" * 100
    )


    if column not in left_df.columns:

        raise ValueError(
            f"{column} missing from {left_name}."
        )


    if column not in right_df.columns:

        raise ValueError(
            f"{column} missing from {right_name}."
        )


    left_part = (

        left_df[
            [
                "SEQN",
                column
            ]
        ]

        .rename(
            columns={
                column:
                    f"{column}_LEFT"
            }
        )

    )


    right_part = (

        right_df[
            [
                "SEQN",
                column
            ]
        ]

        .rename(
            columns={
                column:
                    f"{column}_RIGHT"
            }
        )

    )


    comparison = (
        left_part
        .merge(
            right_part,
            on="SEQN",
            how="outer",
            validate="one_to_one",
            indicator=True
        )
    )


    comparison[
        f"{column}_LEFT"
    ] = (
        comparison[
            f"{column}_LEFT"
        ]
        .fillna("")
    )


    comparison[
        f"{column}_RIGHT"
    ] = (
        comparison[
            f"{column}_RIGHT"
        ]
        .fillna("")
    )


    only_left = int(
        comparison["_merge"]
        .eq("left_only")
        .sum()
    )


    only_right = int(
        comparison["_merge"]
        .eq("right_only")
        .sum()
    )


    print(
        "SEQN only in",
        left_name,
        ":",
        f"{only_left:,}"
    )


    print(
        "SEQN only in",
        right_name,
        ":",
        f"{only_right:,}"
    )


    if require_same_seqn_set:

        if (
            only_left != 0
            or
            only_right != 0
        ):

            raise ValueError(
                f"{left_name} and {right_name} "
                f"do not contain the same SEQN set "
                f"for {column} validation."
            )


    left_raw = (
        comparison[
            f"{column}_LEFT"
        ]
    )


    right_raw = (
        comparison[
            f"{column}_RIGHT"
        ]
    )


    left_populated = (
        left_raw.ne("")
    )


    right_populated = (
        right_raw.ne("")
    )


    both_populated = (
        left_populated
        &
        right_populated
    )


    one_side_missing = (
        left_populated
        ^
        right_populated
    )


    both_blank = (
        ~left_populated
        &
        ~right_populated
    )


    overlap_count = int(
        both_populated.sum()
    )


    one_side_missing_count = int(
        one_side_missing.sum()
    )


    print(
        "Both populated:",
        f"{overlap_count:,}"
    )


    print(
        "Both blank:",
        f"{int(both_blank.sum()):,}"
    )


    print(
        "One populated / one blank:",
        f"{one_side_missing_count:,}"
    )


    # ---------------------------------------------------------------------------------------------
    # NUMERIC CONVERSION FOR COMPARISON ONLY
    # ---------------------------------------------------------------------------------------------

    left_numeric = pd.to_numeric(

        left_raw.loc[
            both_populated
        ],

        errors="raise"

    )


    right_numeric = pd.to_numeric(

        right_raw.loc[
            both_populated
        ],

        errors="raise"

    )


    numeric_equal = np.isclose(

        left_numeric.to_numpy(
            dtype=float
        ),

        right_numeric.to_numpy(
            dtype=float
        ),

        rtol=0,

        atol=0

    )


    numeric_mismatch_count = int(
        (~numeric_equal)
        .sum()
    )


    # ---------------------------------------------------------------------------------------------
    # TEXT DIFFERENCES ARE REPORTED BUT NOT CONSIDERED SCIENTIFIC MISMATCHES
    # ---------------------------------------------------------------------------------------------

    left_text = (

        left_raw.loc[
            both_populated
        ]

        .reset_index(
            drop=True
        )

    )


    right_text = (

        right_raw.loc[
            both_populated
        ]

        .reset_index(
            drop=True
        )

    )


    text_difference_count = int(

        (
            left_text
            !=
            right_text
        )

        .sum()

    )


    print(
        "Raw text-format differences:",
        f"{text_difference_count:,}"
    )


    print(
        "TRUE numeric mismatches:",
        f"{numeric_mismatch_count:,}"
    )


    if numeric_mismatch_count != 0:

        numeric_rows = (

            comparison.loc[
                both_populated
            ]

            .reset_index(
                drop=True
            )

        )


        numeric_rows[
            f"{column}_LEFT_NUMERIC"
        ] = (
            left_numeric
            .reset_index(
                drop=True
            )
        )


        numeric_rows[
            f"{column}_RIGHT_NUMERIC"
        ] = (
            right_numeric
            .reset_index(
                drop=True
            )
        )


        mismatch_examples = (

            numeric_rows.loc[
                ~numeric_equal,
                [
                    "SEQN",

                    f"{column}_LEFT",

                    f"{column}_RIGHT",

                    f"{column}_LEFT_NUMERIC",

                    f"{column}_RIGHT_NUMERIC"
                ]
            ]

            .head(20)

        )


        print(
            "\nTRUE NUMERIC MISMATCH EXAMPLES:"
        )


        display(
            mismatch_examples
        )


        raise ValueError(
            f"{column} has true numeric differences "
            f"between {left_name} and {right_name}."
        )


    print(
        "PASS: Numeric values agree on all "
        "overlapping populated observations."
    )


    return {

        "overlap": overlap_count,

        "text_differences":
            text_difference_count,

        "numeric_mismatches":
            numeric_mismatch_count,

        "one_side_missing":
            one_side_missing_count,

        "only_left_seqn":
            only_left,

        "only_right_seqn":
            only_right

    }


# =================================================================================================
# 10. VALIDATE WTSAF2YR
# =================================================================================================

#
# WTSAF2YR occurs in:
#
#   GLU_D
#   TRIGLY_D
#
# It represents the same fasting subsample weight.
#
# We validate NUMERIC equality first.
#
# We then keep GLU_D as the canonical copy.
#

wtsaf_result = (
    compare_numeric_variable_by_seqn(

        glucose,

        trigly,

        "WTSAF2YR",

        "GLU_D",

        "TRIGLY_D",

        require_same_seqn_set=True

    )
)


#
# For the shared fasting weight, we also require missingness
# to agree by SEQN.
#

if (
    wtsaf_result[
        "one_side_missing"
    ]
    != 0
):

    raise ValueError(
        "WTSAF2YR missing-value positions differ "
        "between GLU_D and TRIGLY_D."
    )


print(
    "\nPASS: WTSAF2YR validated."
)

print(
    "Canonical WTSAF2YR will be retained from GLU_D."
)


# =================================================================================================
# 11. REMOVE REDUNDANT WTSAF2YR FROM TRIGLY_D
# =================================================================================================

trigly_for_merge = (
    trigly.drop(
        columns=[
            "WTSAF2YR"
        ]
    )
)


print(
    "\nRemoved redundant source copy before merge:"
)

print(
    "TRIGLY_D.WTSAF2YR"
)


# =================================================================================================
# 12. VALIDATE PHAFSTHR BETWEEN GLU_D AND FASTQX_D
# =================================================================================================

phafsthr_result = (
    compare_numeric_variable_by_seqn(

        glucose,

        fasting,

        "PHAFSTHR",

        "GLU_D",

        "FASTQX_D",

        require_same_seqn_set=False

    )
)


# =================================================================================================
# 13. VALIDATE PHAFSTMN BETWEEN GLU_D AND FASTQX_D
# =================================================================================================

phafstmn_result = (
    compare_numeric_variable_by_seqn(

        glucose,

        fasting,

        "PHAFSTMN",

        "GLU_D",

        "FASTQX_D",

        require_same_seqn_set=False

    )
)


#
# We intentionally DO NOT require missing positions to match.
#
# FASTQX_D covers more participants.
#
# Therefore one-side-missing observations are expected.
#

print(
    "\nPASS: GLU_D / FASTQX_D fasting-duration "
    "overlap validated numerically."
)


# =================================================================================================
# 14. RENAME FASTQX_D OVERLAP VARIABLES BEFORE MERGE
# =================================================================================================

#
# Do this explicitly rather than relying on pandas suffixes.
#
# This gives meaningful provenance:
#
#     PHAFSTHR_FAST
#     PHAFSTMN_FAST
#

fasting_for_merge = (
    fasting.rename(
        columns={

            "PHAFSTHR":
                "PHAFSTHR_FAST",

            "PHAFSTMN":
                "PHAFSTMN_FAST"

        }
    )
)


if (
    fasting_for_merge.columns
    .duplicated()
    .any()
):

    raise ValueError(
        "FASTQX_D rename created duplicate columns."
    )


print(
    "\nFASTQX_D source fields renamed:"
)

print(
    "PHAFSTHR -> PHAFSTHR_FAST"
)

print(
    "PHAFSTMN -> PHAFSTMN_FAST"
)


# =================================================================================================
# 15. FUNCTION: SAFE MERGE WITH NO UNEXPECTED OVERLAP
# =================================================================================================

def safe_outer_merge(
    base,
    incoming,
    incoming_name
):

    overlap = sorted(

        (
            set(
                base.columns
            )

            &

            set(
                incoming.columns
            )
        )

        -

        {
            "SEQN"
        }

    )


    if overlap:

        raise ValueError(
            f"Unexpected overlapping columns before "
            f"merging {incoming_name}:\n"
            f"{overlap}"
        )


    result = (
        base.merge(

            incoming,

            on="SEQN",

            how="outer",

            validate="one_to_one"

        )
    )


    return result


# =================================================================================================
# 16. EXPECTED FINAL SEQN UNION
# =================================================================================================

expected_seqn_union = set()


for dataset in datasets.values():

    expected_seqn_union.update(
        dataset["SEQN"]
    )


print(
    "\nExpected final unique Lab SEQN:",
    f"{len(expected_seqn_union):,}"
)


# =================================================================================================
# 17. START WITH GLU_D
# =================================================================================================

df = glucose.copy()


# =================================================================================================
# 18. MERGE TCHOL_D
# =================================================================================================

df = safe_outer_merge(

    df,

    chol,

    "TCHOL_D"

)


print(
    "Merged TCHOL_D"
)


# =================================================================================================
# 19. MERGE FASTQX_D
# =================================================================================================

df = safe_outer_merge(

    df,

    fasting_for_merge,

    "FASTQX_D"

)


print(
    "Merged FASTQX_D"
)


# =================================================================================================
# 20. MERGE GHB_D
# =================================================================================================

df = safe_outer_merge(

    df,

    ghb,

    "GHB_D"

)


print(
    "Merged GHB_D"
)


# =================================================================================================
# 21. MERGE HDL_D
# =================================================================================================

df = safe_outer_merge(

    df,

    hdl,

    "HDL_D"

)


print(
    "Merged HDL_D"
)


# =================================================================================================
# 22. MERGE TRIGLY_D
# =================================================================================================

df = safe_outer_merge(

    df,

    trigly_for_merge,

    "TRIGLY_D"

)


print(
    "Merged TRIGLY_D"
)


# =================================================================================================
# 23. NORMALIZE OUTER-MERGE MISSING VALUES
# =================================================================================================

#
# The original corrected CSVs use blank strings for missing.
#
# Outer merge introduces NaN for participants absent from a
# particular source.
#
# Convert those merge-generated NaNs back to blank strings.
#

df = df.fillna("")


# =================================================================================================
# 24. FINAL SEQN VALIDATION
# =================================================================================================

print(
    "\n"
    + "=" * 110
)

print(
    "FINAL SEQN CHECK"
)

print(
    "=" * 110
)


print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Unique SEQN:",
    f"{df['SEQN'].nunique():,}"
)


if (
    len(df)
    !=
    len(expected_seqn_union)
):

    raise ValueError(
        "Final Lab row count differs from "
        "union of source SEQN."
    )


if (
    df["SEQN"].nunique()
    !=
    len(expected_seqn_union)
):

    raise ValueError(
        "Final unique SEQN count differs from "
        "source SEQN union."
    )


if (
    df["SEQN"]
    .duplicated()
    .any()
):

    raise ValueError(
        "Final Lab dataset contains duplicate SEQN."
    )


if (
    df["SEQN"]
    .eq("")
    .any()
):

    raise ValueError(
        "Final Lab dataset contains blank SEQN."
    )


if (
    df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final Lab dataset contains SEQN ending '.0'."
    )


print(
    "PASS: Final SEQN integrity."
)


# =================================================================================================
# 25. DUPLICATE COLUMN NAME CHECK
# =================================================================================================

duplicate_column_names = (

    df.columns[
        df.columns.duplicated()
    ]

    .tolist()

)


if duplicate_column_names:

    raise ValueError(
        "Duplicate column names found:\n"
        + str(
            duplicate_column_names
        )
    )


print(
    "PASS: No duplicate column names."
)


# =================================================================================================
# 26. VERIFY CANONICAL WTSAF2YR
# =================================================================================================

if "WTSAF2YR" not in df.columns:

    raise ValueError(
        "Canonical WTSAF2YR is missing."
    )


BAD_WEIGHT_COLUMNS = [

    "WTSAF2YR_LBXGH",

    "WTSAF2YR_TRIGLY",

    "WTSAF2YR_X",

    "WTSAF2YR_Y",

    "WTSAF2YR_CHOL",

    "WTSAF2YR_HDL"

]


bad_weight_columns_found = [

    col

    for col in BAD_WEIGHT_COLUMNS

    if col in df.columns

]


if bad_weight_columns_found:

    raise ValueError(
        "Redundant WTSAF2YR columns remain:\n"
        + str(
            bad_weight_columns_found
        )
    )


print(
    "PASS: Exactly one canonical WTSAF2YR retained."
)


# =================================================================================================
# 27. VERIFY FASTQX_D FASTING FIELDS
# =================================================================================================

EXPECTED_FASTING_FIELDS = [

    "PHAFSTHR",

    "PHAFSTMN",

    "PHAFSTHR_FAST",

    "PHAFSTMN_FAST"

]


for col in EXPECTED_FASTING_FIELDS:

    if col not in df.columns:

        raise ValueError(
            f"Expected fasting variable missing: {col}"
        )


print(
    "PASS: Both GLU_D and FASTQX_D "
    "fasting-duration fields preserved."
)


# =================================================================================================
# 28. VERIFY NO MISLEADING MERGE SUFFIXES
# =================================================================================================

#
# _FAST is intentional.
#
# The others below should never be created now.
#

BAD_SUFFIXES = [

    "_LBXGH",

    "_TRIGLY",

    "_CHOL",

    "_HDL",

    "_X",

    "_Y"

]


unexpected_suffix_columns = []


for col in df.columns:

    if any(
        col.endswith(
            suffix
        )
        for suffix in BAD_SUFFIXES
    ):

        unexpected_suffix_columns.append(
            col
        )


if unexpected_suffix_columns:

    raise ValueError(
        "Unexpected merge-generated suffix columns:\n"
        + str(
            unexpected_suffix_columns
        )
    )


print(
    "PASS: No misleading merge-generated suffix columns."
)


# =================================================================================================
# 29. FUNCTION: EXACT SOURCE TEXT FIDELITY
# =================================================================================================

#
# For retained source-specific columns, ensure the exact source
# CSV text survived the merge.
#

def validate_source_text_fidelity(
    source_df,
    final_df,
    source_name,
    column_mapping
):

    print(
        f"\n--- SOURCE FIDELITY: {source_name} ---"
    )


    source_columns = (
        ["SEQN"]
        +
        list(
            column_mapping.keys()
        )
    )


    final_columns = (
        ["SEQN"]
        +
        list(
            dict.fromkeys(
                column_mapping.values()
            )
        )
    )


    source_check = (
        source_df[
            source_columns
        ]
        .copy()
    )


    final_check = (
        final_df[
            final_columns
        ]
        .copy()
    )


    final_rename = {}


    for source_col, final_col in column_mapping.items():

        final_rename[
            final_col
        ] = (
            source_col
            + "__FINAL"
        )


    final_check = (
        final_check.rename(
            columns=final_rename
        )
    )


    check = (
        source_check.merge(

            final_check,

            on="SEQN",

            how="left",

            validate="one_to_one"

        )
    )


    check = (
        check.fillna("")
    )


    failures = []


    for source_col in column_mapping:

        final_col = (
            source_col
            + "__FINAL"
        )


        mismatch_count = int(

            (
                check[
                    source_col
                ]

                !=

                check[
                    final_col
                ]
            )

            .sum()

        )


        if mismatch_count > 0:

            failures.append(
                (
                    source_col,
                    mismatch_count
                )
            )


    if failures:

        raise ValueError(
            f"{source_name} source-text fidelity "
            f"failures:\n{failures}"
        )


    print(
        "PASS:",
        len(column_mapping),
        "retained source variables match exactly."
    )


# =================================================================================================
# 30. GLU_D SOURCE FIDELITY
# =================================================================================================

GLU_MAPPING = {

    col: col

    for col in glucose.columns

    if col != "SEQN"

}


validate_source_text_fidelity(

    glucose,

    df,

    "GLU_D",

    GLU_MAPPING

)


# =================================================================================================
# 31. TCHOL_D SOURCE FIDELITY
# =================================================================================================

TCHOL_MAPPING = {

    col: col

    for col in chol.columns

    if col != "SEQN"

}


validate_source_text_fidelity(

    chol,

    df,

    "TCHOL_D",

    TCHOL_MAPPING

)


# =================================================================================================
# 32. FASTQX_D SOURCE FIDELITY
# =================================================================================================

FASTQX_MAPPING = {}


for col in fasting.columns:

    if col == "SEQN":

        continue


    if col == "PHAFSTHR":

        FASTQX_MAPPING[
            col
        ] = (
            "PHAFSTHR_FAST"
        )


    elif col == "PHAFSTMN":

        FASTQX_MAPPING[
            col
        ] = (
            "PHAFSTMN_FAST"
        )


    else:

        FASTQX_MAPPING[
            col
        ] = col


validate_source_text_fidelity(

    fasting,

    df,

    "FASTQX_D",

    FASTQX_MAPPING

)


# =================================================================================================
# 33. GHB_D SOURCE FIDELITY
# =================================================================================================

GHB_MAPPING = {

    col: col

    for col in ghb.columns

    if col != "SEQN"

}


validate_source_text_fidelity(

    ghb,

    df,

    "GHB_D",

    GHB_MAPPING

)


# =================================================================================================
# 34. HDL_D SOURCE FIDELITY
# =================================================================================================

HDL_MAPPING = {

    col: col

    for col in hdl.columns

    if col != "SEQN"

}


validate_source_text_fidelity(

    hdl,

    df,

    "HDL_D",

    HDL_MAPPING

)


# =================================================================================================
# 35. TRIGLY_D SOURCE FIDELITY
# =================================================================================================

#
# WTSAF2YR was intentionally deduplicated and was already
# validated numerically against GLU_D.
#
# All remaining TRIGLY_D fields must survive textually exactly.
#

TRIGLY_MAPPING = {

    col: col

    for col in trigly.columns

    if (
        col != "SEQN"
        and
        col != "WTSAF2YR"
    )

}


validate_source_text_fidelity(

    trigly,

    df,

    "TRIGLY_D",

    TRIGLY_MAPPING

)


# =================================================================================================
# 36. REPRESENTATIVE DECIMAL PROTECTION CHECK
# =================================================================================================

#
# This is diagnostic only.
#
# We are NOT converting or rounding any values.
#

REPRESENTATIVE_DECIMAL_FIELDS = [

    "WTSAF2YR",

    "LBDGLUSI",

    "LBXIN",

    "LBDINSI",

    "LBXGH",

    "LBDHDDSI",

    "LBDTRSI",

    "LBDLDLSI",

    "LBDTCSI"

]


print(
    "\n"
    + "=" * 110
)

print(
    "LAB DECIMAL PRESERVATION"
)

print(
    "=" * 110
)


for col in REPRESENTATIVE_DECIMAL_FIELDS:

    if col not in df.columns:

        continue


    populated_count = int(
        df[col]
        .ne("")
        .sum()
    )


    decimal_token_count = int(

        (
            df[col].ne("")
            &
            df[col]
            .str.contains(
                ".",
                regex=False
            )
        )

        .sum()

    )


    print(
        f"{col}: "
        f"populated={populated_count:,}, "
        f"decimal tokens={decimal_token_count:,}"
    )


print(
    "PASS: No Lab field was rounded or "
    "integerized during merge."
)


# =================================================================================================
# 37. TINY XPORT ARTIFACT CHECK
# =================================================================================================

TINY_TOKEN = (
    "5.397605346934028e-79"
)


tiny_artifact_count = 0


tiny_artifact_columns = {}


for col in df.columns:

    if col == "SEQN":

        continue


    count = int(

        df[col]
        .str.lower()
        .eq(
            TINY_TOKEN.lower()
        )
        .sum()

    )


    if count > 0:

        tiny_artifact_columns[
            col
        ] = count


        tiny_artifact_count += count


print(
    "\nTiny XPORT artifact count:",
    tiny_artifact_count
)


if tiny_artifact_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in merged Lab data:\n"
        + str(
            tiny_artifact_columns
        )
    )


print(
    "PASS: No known XPORT tiny-zero artifacts."
)


# =================================================================================================
# 38. SAVE FINAL LAB CSV
# =================================================================================================

df.to_csv(

    OUTPUT_PATH,

    index=False

)


print(
    "\nSaved:"
)

print(
    OUTPUT_PATH
)


# =================================================================================================
# 39. RE-READ FINAL CSV AS TEXT
# =================================================================================================

final_df = pd.read_csv(

    OUTPUT_PATH,

    dtype=str,

    keep_default_na=False

)


# =================================================================================================
# 40. POST-SAVE STRUCTURE CHECK
# =================================================================================================

if (
    final_df.shape
    !=
    df.shape
):

    raise ValueError(
        "CSV shape changed after save."
    )


if (
    final_df.columns.tolist()
    !=
    df.columns.tolist()
):

    raise ValueError(
        "CSV column structure changed after save."
    )


print(
    "PASS: CSV structure preserved."
)


# =================================================================================================
# 41. EXACT PRE-SAVE / POST-SAVE TEXT FIDELITY
# =================================================================================================

post_save_failures = []


for col in df.columns:

    same = (

        df[col]
        .eq(
            final_df[col]
        )
        .all()

    )


    if not same:

        post_save_failures.append(
            col
        )


if post_save_failures:

    raise ValueError(
        "Text changed during CSV export:\n"
        + str(
            post_save_failures
        )
    )


print(
    "PASS: Exact pre-save/post-save text fidelity."
)


# =================================================================================================
# 42. FINAL SEQN CSV CHECK
# =================================================================================================

if (
    final_df["SEQN"]
    .duplicated()
    .any()
):

    raise ValueError(
        "Saved CSV contains duplicate SEQN."
    )


if (
    final_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Saved CSV contains SEQN '.0'."
    )


if (
    final_df["SEQN"]
    .eq("")
    .any()
):

    raise ValueError(
        "Saved CSV contains blank SEQN."
    )


print(
    "PASS: Saved CSV SEQN integrity."
)


# =================================================================================================
# 43. FINAL WTSAF2YR CHECK
# =================================================================================================

final_weight_columns = [

    col

    for col in final_df.columns

    if col.startswith(
        "WTSAF2YR"
    )

]


print(
    "\nWTSAF2YR fields in final CSV:"
)

print(
    final_weight_columns
)


if final_weight_columns != [
    "WTSAF2YR"
]:

    raise ValueError(
        "Expected exactly one WTSAF2YR field."
    )


print(
    "PASS: One canonical WTSAF2YR."
)


# =================================================================================================
# 44. FINAL FASTING FIELD CHECK
# =================================================================================================

for col in [

    "PHAFSTHR",

    "PHAFSTMN",

    "PHAFSTHR_FAST",

    "PHAFSTMN_FAST"

]:

    if col not in final_df.columns:

        raise ValueError(
            f"Missing final fasting field: {col}"
        )


print(
    "PASS: GLU_D and FASTQX_D fasting fields retained."
)


# =================================================================================================
# 45. FINAL SUMMARY
# =================================================================================================

print(
    "\n"
    + "=" * 110
)

print(
    "NHANES 2005-2006 LAB MERGE COMPLETE"
)

print(
    "=" * 110
)


print(
    "Rows:",
    f"{len(final_df):,}"
)


print(
    "Columns:",
    f"{len(final_df.columns):,}"
)


print(
    "Unique SEQN:",
    f"{final_df['SEQN'].nunique():,}"
)


print(
    ""
)


print(
    "WTSAF2YR:"
)


print(
    "  GLU_D canonical copy retained"
)


print(
    "  TRIGLY_D duplicate validated then removed"
)


print(
    "  WTSAF2YR_LBXGH not created"
)


print(
    ""
)


print(
    "PHAFSTHR / PHAFSTMN:"
)


print(
    "  GLU_D versions retained:"
)


print(
    "    PHAFSTHR"
)


print(
    "    PHAFSTMN"
)


print(
    "  FASTQX_D versions retained:"
)


print(
    "    PHAFSTHR_FAST"
)


print(
    "    PHAFSTMN_FAST"
)


print(
    ""
)


print(
    "PHAFSTHR overlapping numeric mismatches:",
    phafsthr_result[
        "numeric_mismatches"
    ]
)


print(
    "PHAFSTHR text-format differences:",
    phafsthr_result[
        "text_differences"
    ]
)


print(
    "PHAFSTMN overlapping numeric mismatches:",
    phafstmn_result[
        "numeric_mismatches"
    ]
)


print(
    "PHAFSTMN text-format differences:",
    phafstmn_result[
        "text_differences"
    ]
)


print(
    ""
)


print(
    "Duplicate SEQN: 0"
)


print(
    "SEQN '.0': 0"
)


print(
    "Duplicate column names: 0"
)


print(
    "Known tiny XPORT artifacts: 0"
)


print(
    "Lab decimal reformatting during merge: 0"
)


print(
    "Source-specific retained-field fidelity: PASS"
)


print(
    "Pre-save/post-save text fidelity: PASS"
)


print(
    "=" * 110
)

NHANES 2011-2012 LAB MERGE


,file,rows,columns,column_names
0,GLU_D,3329,6,"[SEQN, WTSAF2YR, LBXGLU, LBDGLUSI, PHAFSTHR, P..."
1,TCHOL_D,8291,3,"[SEQN, LBXTC, LBDTCSI]"
2,FASTQX_D,9422,19,"[SEQN, PHQ020, PHACOFHR, PHACOFMN, PHQ030, PHA..."
3,GHB_D,6979,2,"[SEQN, LBXGH]"
4,HDL_D,8291,3,"[SEQN, LBDHDD, LBDHDDSI]"
5,TRIGLY_D,3329,6,"[SEQN, WTSAF2YR, LBXTR, LBDTRSI, LBDLDL, LBDLD..."



FILE: GLU_D
ROWS: 3,329
COLUMNS: 6
['SEQN', 'WTSAF2YR', 'LBXGLU', 'LBDGLUSI', 'PHAFSTHR', 'PHAFSTMN']

FILE: TCHOL_D
ROWS: 8,291
COLUMNS: 3
['SEQN', 'LBXTC', 'LBDTCSI']

FILE: FASTQX_D
ROWS: 9,422
COLUMNS: 19
['SEQN', 'PHQ020', 'PHACOFHR', 'PHACOFMN', 'PHQ030', 'PHAALCHR', 'PHAALCMN', 'PHQ040', 'PHAGUMHR', 'PHAGUMMN', 'PHQ050', 'PHAANTHR', 'PHAANTMN', 'PHQ060', 'PHASUPHR', 'PHASUPMN', 'PHAFSTHR', 'PHAFSTMN', 'PHDSESN']

FILE: GHB_D
ROWS: 6,979
COLUMNS: 2
['SEQN', 'LBXGH']

FILE: HDL_D
ROWS: 8,291
COLUMNS: 3
['SEQN', 'LBDHDD', 'LBDHDDSI']

FILE: TRIGLY_D
ROWS: 3,329
COLUMNS: 6
['SEQN', 'WTSAF2YR', 'LBXTR', 'LBDTRSI', 'LBDLDL', 'LBDLDLSI']

--- SEQN CHECK: GLU_D ---
Rows: 3,329
Unique SEQN: 3,329
Blank SEQN: 0
Duplicate SEQN: 0
SEQN ending '.0': 0
Invalid SEQN format: 0
PASS

--- SEQN CHECK: TCHOL_D ---
Rows: 8,291
Unique SEQN: 8,291
Blank SEQN: 0
Duplicate SEQN: 0
SEQN ending '.0': 0
Invalid SEQN format: 0
PASS

--- SEQN CHECK: FASTQX_D ---
Rows: 9,422
Unique SEQN: 9,422
Blank SEQN: 0
